In [ ]:
import os
import gc
import re
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, InputLayer
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split, LeaveOneGroupOut
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, log_loss, confusion_matrix)

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.family": "DejaVu Sans"})
ACCENT, ACCENT2, RED, GREEN, AMBER = "#7C3AED", "#06B6D4", "#EF4444", "#22C55E", "#F59E0B"

In [ ]:
# ============================================================
# 0 -- Config
# ============================================================
# Phase 1, driven by the Phase 0 diagnostics (signer_diagnostics.ipynb):
#   * 41 exact + 50 near-duplicate pairs, ALL same-signer  -> de-duplicate.
#     (LOSO was never affected by these; the stratified split was.)
#   * Signer identity is 99.7% linearly recoverable        -> test delta features.
#   * Signer_B is left-dominant, the other three are right-dominant, and
#     mirroring B cuts its distance to their profile by 72%
#                                                          -> test mirror augmentation.
#
# 17 training runs total: 1 stratified reference + 4 arms x 4 LOSO folds.
# Roughly 7 hours at the ~25 min/run measured previously. Run once, overnight.
DATA_DIR        = "features"
SEQUENCE_LENGTH = 30
FEATURES_DIM    = 447
MAX_SAMPLES     = 110
RANDOM_STATE    = 42
BATCH_SIZE      = 32
EPOCHS          = 150

# Frozen Hyperband-selected hyperparameters (unchanged from tuner_results.py).
GRU1, GRU2, DROPOUT, L2_RATE, USE_REC_L2, LR = 32, 256, 0.3, 1e-4, False, 1e-3

POSE_SLICE = slice(0, 99)
FACE_SLICE = slice(99, 321)
LH_SLICE   = slice(321, 384)
RH_SLICE   = slice(384, 447)

NEAR_DUP_REL_THRESHOLD = 0.05   # matches Phase 0
MIRROR_PROB = 0.5               # per-sample random horizontal flip during training

# Arms to run. Each is (feature_variant, use_mirror, use_delta).
# Trim this list to shorten the run; the stratified reference below is separate.
ARMS = [
    ("Hand-only",             ["LH", "RH"],                    True,  False),
    ("Hand+Pose",             ["POSE", "LH", "RH"],            True,  False),
    ("Full",                  ["POSE", "FACE", "LH", "RH"],    True,  False),
    ("Full + delta",          ["POSE", "FACE", "LH", "RH"],    True,  True),
]
SLICE_MAP = {"POSE": POSE_SLICE, "FACE": FACE_SLICE, "LH": LH_SLICE, "RH": RH_SLICE}


# NMS-focused / standard-manual split -- UNVERIFIED RECONSTRUCTION, carried
# over from ablation_loso_study.py. The repo records no ground-truth list;
# this was reconstructed from cross-linguistic NMS conventions and a
# motion-magnitude check that came back inconclusive. Exploratory only.
NMS_FOCUSED_CLASSES = {
    "Apa", "Bagaimana", "Berapa", "Dimana", "Kapan", "Kemana", "Siapa",
    "Bingung", "Marah", "Ramah", "Sabar", "Sedih", "Senang", "Baik",
    "Apa Kabar", "Halo", "Terima Kasih", "Tinggi", "Pendek", "Melihat",
}


def classify_signer(fname: str) -> str:
    stem = re.sub(r"\s*\(\d+\)$", "", os.path.splitext(fname)[0])
    if stem.upper().startswith("BISINDO_"):
        return "Signer_D_bisindo"
    if re.fullmatch(r"\d+", stem):
        return "Signer_A_numeric"
    if re.search(r"-\d+$", stem):
        return "Signer_B_dash"
    if re.search(r"_\d+$", stem):
        return "Signer_C_underscore"
    return "Signer_UNK"

In [ ]:
# ============================================================
# 1 -- Load, DE-DUPLICATE, then balance
# ============================================================
# Order matters: de-duplicating before balancing means the 110/class quota is
# filled with 110 genuinely distinct sequences rather than counting copies.
print("Loading dataset...")
actions = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
num_classes = len(actions)

sequences, labels, groups, fnames = [], [], [], []
for i, action in enumerate(actions):
    action_path = os.path.join(DATA_DIR, action)
    for f in tqdm([f for f in os.listdir(action_path) if f.endswith(".npy")],
                  desc=f"  {action:<15}", leave=False):
        seq = np.load(os.path.join(action_path, f))
        if seq.shape == (SEQUENCE_LENGTH, FEATURES_DIM):
            sequences.append(seq); labels.append(i)
            groups.append(classify_signer(f)); fnames.append(f"{action}/{f}")

X = np.array(sequences, dtype=np.float32)
y_int = np.array(labels); group_all = np.array(groups); fname_all = np.array(fnames)
print(f"  Raw: {len(X)} sequences")

# Drop exact and near-duplicates, keeping the first occurrence of each cluster.
flat = X.reshape(len(X), -1)
norms = np.linalg.norm(flat, axis=1)
sq = norms[:, None] ** 2 + norms[None, :] ** 2 - 2 * (flat @ flat.T)
np.maximum(sq, 0, out=sq)
rel = np.sqrt(sq, out=sq) / np.maximum((norms[:, None] + norms[None, :]) / 2.0, 1e-8)
np.fill_diagonal(rel, np.inf)

drop = np.zeros(len(X), dtype=bool)
for i in range(len(X)):
    if drop[i]:
        continue
    dupes = np.where((rel[i] < NEAR_DUP_REL_THRESHOLD) & (~drop))[0]
    drop[dupes[dupes > i]] = True
del flat, sq, rel
gc.collect()

keep = ~drop
X, y_int, group_all, fname_all = X[keep], y_int[keep], group_all[keep], fname_all[keep]
print(f"  De-duplicated: removed {int(drop.sum())}, {len(X)} remain")

idx = []
for i in range(num_classes):
    idx.extend(np.where(y_int == i)[0][:MAX_SAMPLES])
X, y_int, group_all, fname_all = X[idx], y_int[idx], group_all[idx], fname_all[idx]
y_oh = to_categorical(y_int, num_classes=num_classes).astype(np.float32)
print(f"  Balanced: {X.shape[0]} sequences across {num_classes} classes")
print("  Per-signer:", {g: int((group_all == g).sum()) for g in sorted(set(group_all.tolist()))})

In [ ]:
# ============================================================
# 2 -- Mirror mapping, with an empirical self-check
# ============================================================
# A horizontal mirror needs three things: negate every x, swap the two hand
# blocks, and swap each left/right landmark pair within pose and face.
#
# The pose pair list is the standard MediaPipe Pose topology (recalled, not
# read from the library), so it is VERIFIED empirically below rather than
# trusted. Pose is emitted in natural landmark order 0..32 by extraction.py,
# so these index pairs apply directly.
POSE_PAIRS = [(1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12), (13, 14),
              (15, 16), (17, 18), (19, 20), (21, 22), (23, 24), (25, 26),
              (27, 28), (29, 30), (31, 32)]

SELECTED_FACE_IDS = [
    0, 13, 14, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84, 87, 88, 91, 95, 146,
    178, 181, 191, 267, 269, 270, 291, 308, 310, 311, 312, 314, 317, 318,
    321, 324, 375, 402, 405, 415,
    46, 52, 53, 55, 65, 70, 105, 107, 276, 282, 283, 285, 295, 300, 334, 336,
    50, 118, 123, 137, 205, 206, 207, 212, 214, 216,
    280, 347, 352, 366, 425, 426, 427, 432, 434, 436,
]
# CRITICAL ORDERING NOTE. extraction.py emits face landmarks in ASCENDING
# MEDIAPIPE ID order, because it filters with
#     for i, lm in enumerate(lm_list.landmark):
#         if is_face and i not in SELECTED_FACE_IDS: continue
# so the feature block follows sorted(SELECTED_FACE_IDS), NOT the grouped
# order this list is written in. Pairing must therefore be built in id space
# and then remapped to sorted positions. Assuming list order silently
# scrambles face landmarks across the midline (verified failure: it pairs
# lip points with eyebrow points).
#
# The pairing itself is read off this list's own structure -- each group
# writes its left members first, then its right members in matching order --
# so it does not depend on recalled FaceMesh topology. List positions 0..3
# (ids 0, 13, 14, 17) are midline and stay in place.
FACE_LIST_GROUPS = [(4, 17), (38, 8), (54, 10)]  # (start_of_left, n_per_side)

SORTED_FACE_IDS = sorted(SELECTED_FACE_IDS)
assert len(SORTED_FACE_IDS) == 74 and len(set(SORTED_FACE_IDS)) == 74
_face_pos = {lm_id: p for p, lm_id in enumerate(SORTED_FACE_IDS)}

FACE_PERM = np.arange(74)
for _start, _n in FACE_LIST_GROUPS:
    for _k in range(_n):
        _a = _face_pos[SELECTED_FACE_IDS[_start + _k]]
        _b = _face_pos[SELECTED_FACE_IDS[_start + _n + _k]]
        FACE_PERM[_a], FACE_PERM[_b] = _b, _a


def mirror_batch(A):
    """Horizontally mirror (N, 30, 447) landmark sequences."""
    M = A.copy()
    M[:, :, 0::3] *= -1.0                                   # negate all x

    pose = M[:, :, POSE_SLICE].reshape(len(M), SEQUENCE_LENGTH, 33, 3)
    for a, b in POSE_PAIRS:
        pose[:, :, [a, b]] = pose[:, :, [b, a]]
    M[:, :, POSE_SLICE] = pose.reshape(len(M), SEQUENCE_LENGTH, 99)

    face = M[:, :, FACE_SLICE].reshape(len(M), SEQUENCE_LENGTH, 74, 3)
    M[:, :, FACE_SLICE] = face[:, :, FACE_PERM].reshape(len(M), SEQUENCE_LENGTH, 222)

    lh = M[:, :, LH_SLICE].copy()
    M[:, :, LH_SLICE] = M[:, :, RH_SLICE]
    M[:, :, RH_SLICE] = lh
    return M


# Self-check 1: mirroring twice must be the identity.
_probe = X[:8]
assert np.allclose(mirror_batch(mirror_batch(_probe)), _probe, atol=1e-5), \
    "mirror_batch is not an involution -- the pair mapping is wrong"

# Self-check 2: mirroring must move Signer_B's hand-presence profile toward
# the other three (the Phase 0 finding). This validates the mapping against
# real data rather than against my assumptions about landmark ordering.
def _hand_profile(A):
    return np.array([float((~np.all(A[:, :, LH_SLICE] == 0, axis=-1)).mean()),
                     float((~np.all(A[:, :, RH_SLICE] == 0, axis=-1)).mean())])


_B = X[group_all == "Signer_B_dash"]
_oth = X[(group_all != "Signer_B_dash") & (group_all != "Signer_UNK")]
_d_raw = np.abs(_hand_profile(_B) - _hand_profile(_oth)).sum()
_d_mir = np.abs(_hand_profile(mirror_batch(_B)) - _hand_profile(_oth)).sum()
print(f"\nMirror self-check: Signer_B distance to others {_d_raw:.3f} -> {_d_mir:.3f} when mirrored")
assert _d_mir < _d_raw, "mirroring did not align Signer_B -- check the mapping"

# Self-check 3: the landmark PAIRINGS themselves. Checks 1 and 2 both pass
# even when the pairing is wrong (any swap is an involution, and the
# hand-block swap alone realigns Signer_B), so this is the one that actually
# catches a mis-specified pair table.
#   x: a partner sits at the opposite horizontal offset -> corr(x[p], -x) ~ +1
#   y: a partner sits at the SAME height                -> corr(y[p],  y) ~ +1
# The y test is essential. A wrong table that pairs lips with eyebrows still
# scores ~+0.99 on x alone, and only the height check rejects it.
_pose_perm = np.arange(33)
for _a, _b in POSE_PAIRS:
    _pose_perm[_a], _pose_perm[_b] = _pose_perm[_b], _pose_perm[_a]

for _label, _sl, _n, _perm in [("pose", POSE_SLICE, 33, _pose_perm),
                               ("face", FACE_SLICE, 74, FACE_PERM)]:
    assert np.all(_perm[_perm] == np.arange(_n)), f"{_label} pairing is not an involution"
    _blk = X[:, :, _sl].reshape(len(X), SEQUENCE_LENGTH, _n, 3)
    _present = ~np.all(_blk.reshape(len(X), SEQUENCE_LENGTH, -1) == 0, axis=-1)
    _m = _blk[_present].mean(axis=0)
    _cx = np.corrcoef(_m[:, 0][_perm], -_m[:, 0])[0, 1]
    _cy = np.corrcoef(_m[:, 1][_perm], _m[:, 1])[0, 1]
    _dy = float(np.abs(_m[:, 1] - _m[:, 1][_perm]).mean())
    print(f"  {_label}: corr_x={_cx:+.4f}  corr_y={_cy:+.4f}  mean|dy|={_dy:.4f}")
    assert _cx > 0.9, f"{_label} left/right pairing is wrong (x mismatch)"
    assert _cy > 0.9, f"{_label} left/right pairing is wrong (y/height mismatch)"

print("  OK -- mirror mapping validated (involution + Signer_B + x/y pair geometry)")

In [ ]:
# ============================================================
# 3 -- Feature builders: slicing and delta
# ============================================================
def slice_features(A, names):
    return np.concatenate([A[:, :, SLICE_MAP[n]] for n in names], axis=-1)


def to_delta(A):
    """Frame-to-frame differences, zero-padded at t=0 to preserve 30 frames.
    Drops absolute body/face geometry (the 99.7%-recoverable signer signal)
    while keeping motion."""
    d = np.zeros_like(A)
    d[:, 1:, :] = np.diff(A, axis=1)
    return d

In [ ]:
# ============================================================
# 4 -- Augmentation (train-only, x3, optional mirroring)
# ============================================================
def augment_sequence(data, rng):
    n_frames, n_feat = data.shape
    aug = data.copy().astype(np.float32)
    aug += rng.normal(0, 0.005, aug.shape).astype(np.float32)
    aug = aug.reshape(n_frames, -1, 3)
    aug *= np.float32(rng.uniform(0.95, 1.05))
    angle = np.radians(rng.uniform(3, 5) * rng.choice([-1, 1]))
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=np.float32)
    aug = np.dot(aug, R).reshape(n_frames, n_feat)
    new_len = int(n_frames * rng.uniform(0.9, 1.1))
    out = np.zeros((n_frames, n_feat), dtype=np.float32)
    for f in range(n_feat):
        out[:, f] = np.interp(np.linspace(0, n_frames - 1, n_frames),
                              np.linspace(0, n_frames - 1, new_len),
                              np.interp(np.linspace(0, n_frames - 1, new_len),
                                        np.arange(n_frames), aug[:, f]))
    return out


def build_training_pool(X_tr_raw, y_tr, use_mirror, seed):
    """Mirror (optionally) on the FULL 447-dim vector, then x3 augment.
    Mirroring must happen before feature slicing, because it permutes
    landmark blocks that slicing would have already discarded."""
    rng = np.random.default_rng(seed)
    A = X_tr_raw.copy()
    if use_mirror:
        flip = rng.random(len(A)) < MIRROR_PROB
        if flip.any():
            A[flip] = mirror_batch(A[flip])
    n = len(A)
    Xa = np.empty((n * 3, A.shape[1], A.shape[2]), dtype=np.float32)
    ya = np.empty((n * 3, y_tr.shape[1]), dtype=np.float32)
    Xa[:n], ya[:n] = A, y_tr
    for i, seq in enumerate(A):
        Xa[n + i] = augment_sequence(seq, rng)
        Xa[2 * n + i] = augment_sequence(seq, rng)
    ya[n:2 * n] = y_tr
    ya[2 * n:3 * n] = y_tr
    perm = rng.permutation(len(Xa))
    return Xa[perm], ya[perm]

In [ ]:
# ============================================================
# 5 -- Model builder and train/eval (now records predictions)
# ============================================================
def build_model(input_dim):
    kreg = regularizers.l2(L2_RATE)
    rreg = regularizers.l2(L2_RATE) if USE_REC_L2 else None
    model = Sequential([
        InputLayer(input_shape=(SEQUENCE_LENGTH, input_dim)),
        GRU(GRU1, return_sequences=True, name="gru_1",
            kernel_regularizer=kreg, recurrent_regularizer=rreg),
        Dropout(DROPOUT, name="dropout_1"),
        GRU(GRU2, return_sequences=False, name="gru_2",
            kernel_regularizer=kreg, recurrent_regularizer=rreg),
        Dropout(DROPOUT, name="dropout_2"),
        Dense(num_classes, activation="softmax", name="output",
              kernel_regularizer=regularizers.l2(L2_RATE)),
    ], name="SignLingo_GRU_Phase1")
    model.compile(optimizer=Adam(LR), loss="categorical_crossentropy", metrics=["accuracy"])
    return model


def train_and_eval(X_tr, y_tr, X_val, y_val, X_te, y_te):
    tf.keras.backend.clear_session()
    model = build_model(X_tr.shape[-1])
    model.fit(
        tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
          .shuffle(len(X_tr)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        validation_data=tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        epochs=EPOCHS,
        callbacks=[EarlyStopping(monitor="val_loss", patience=15,
                                 restore_best_weights=True, verbose=0),
                   ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                     patience=5, min_lr=1e-6, verbose=0)],
        verbose=0,
    )
    probs = model.predict(X_te, verbose=0)
    y_pred, y_true = probs.argmax(axis=1), y_te.argmax(axis=1)
    row = {
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "loss":      float(log_loss(y_true, probs, labels=list(range(num_classes)))),
        # Everything below is saved so no question needs a third training run.
        "y_true": y_true.astype(int).tolist(),
        "y_pred": y_pred.astype(int).tolist(),
        "confusion_matrix": confusion_matrix(y_true, y_pred,
                                             labels=list(range(num_classes))).astype(int).tolist(),
    }
    del model
    gc.collect()
    return row

In [ ]:
# ============================================================
# 6 -- Stratified reference run (de-duplicated)
# ============================================================
# One run, for a like-for-like sequence-level number on the cleaned data.
# This is what the paper's 99.32% should be compared against, since that
# figure was measured on a set still containing same-signer duplicates.
print("\n" + "=" * 60)
print("STRATIFIED REFERENCE (de-duplicated, Full features)")
print("=" * 60)
tr_i, tmp_i = train_test_split(np.arange(len(X)), test_size=0.20,
                               stratify=y_int, random_state=RANDOM_STATE)
va_i, te_i = train_test_split(tmp_i, test_size=0.50,
                              stratify=y_int[tmp_i], random_state=RANDOM_STATE)
Xtr, ytr = build_training_pool(X[tr_i], y_oh[tr_i], use_mirror=False, seed=RANDOM_STATE)
strat_row = train_and_eval(slice_features(Xtr, ["POSE", "FACE", "LH", "RH"]), ytr,
                           slice_features(X[va_i], ["POSE", "FACE", "LH", "RH"]), y_oh[va_i],
                           slice_features(X[te_i], ["POSE", "FACE", "LH", "RH"]), y_oh[te_i])
print(f"  accuracy={strat_row['accuracy']*100:.2f}%  f1={strat_row['f1']:.4f}")

In [ ]:
# ============================================================
# 7 -- Main grid: 4 arms x 4 LOSO folds
# ============================================================
known = (group_all != "Signer_UNK")
Xk, yk_int, yk_oh, gk = X[known], y_int[known], y_oh[known], group_all[known]
print(f"\nLOSO pool: {len(Xk)} sequences ({int((~known).sum())} unclassifiable dropped)")

results = {"stratified_reference": strat_row, "arms": {}}
logo = LeaveOneGroupOut()

for arm_name, feats, use_mirror, use_delta in ARMS:
    print("\n" + "=" * 60)
    print(f"ARM: {arm_name}   (mirror={use_mirror}, delta={use_delta})")
    print("=" * 60)
    arm_rows = []
    for tr_idx, te_idx in logo.split(Xk, yk_int, gk):
        held = gk[te_idx][0]
        X_trf, y_trf = Xk[tr_idx], yk_oh[tr_idx]
        X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
            X_trf, y_trf, test_size=0.12, stratify=yk_int[tr_idx], random_state=RANDOM_STATE)

        # Mirror + augment on full vectors, then slice, then optionally delta.
        X_tr_full, y_tr = build_training_pool(X_tr_raw, y_tr, use_mirror, RANDOM_STATE)
        X_tr = slice_features(X_tr_full, feats)
        X_val = slice_features(X_val_raw, feats)
        X_te = slice_features(Xk[te_idx], feats)
        if use_delta:
            X_tr, X_val, X_te = to_delta(X_tr), to_delta(X_val), to_delta(X_te)

        row = train_and_eval(X_tr, y_tr, X_val, y_val, X_te, yk_oh[te_idx])
        row.update({"held_out_signer": held, "n_test": int(len(te_idx)),
                    "n_train_before_aug": int(len(X_tr_raw))})

        # NMS-focused vs manual breakdown (see signer_diagnostics notes: the
        # class list is an unverified reconstruction, treat as exploratory).
        yt = np.array(row["y_true"]); yp = np.array(row["y_pred"])
        is_nms = np.array([actions[i] in NMS_FOCUSED_CLASSES for i in yt])
        row["accuracy_nms_focused"] = float(accuracy_score(yt[is_nms], yp[is_nms])) if is_nms.any() else None
        row["accuracy_manual"] = float(accuracy_score(yt[~is_nms], yp[~is_nms])) if (~is_nms).any() else None

        arm_rows.append(row)
        print(f"  {held:<22} n_test={row['n_test']:<5} acc={row['accuracy']*100:6.2f}%  "
              f"f1={row['f1']:.4f}")
        del X_tr, X_val, X_te, X_tr_full
        gc.collect()

    accs = np.array([r["accuracy"] for r in arm_rows])
    results["arms"][arm_name] = {
        "config": {"features": feats, "mirror": use_mirror, "delta": use_delta},
        "folds": arm_rows,
        "mean_accuracy": float(accs.mean()), "std_accuracy": float(accs.std()),
    }
    print(f"  -> mean {accs.mean()*100:.2f}% +/- {accs.std()*100:.2f}%")

with open("phase1_results.json", "w") as fh:
    json.dump(results, fh, indent=2)
print("\nSaved -> phase1_results.json")

In [ ]:
# ============================================================
# 8 -- Summary vs the Phase 0 baseline
# ============================================================
print("\n" + "=" * 60)
print("PHASE 1 SUMMARY")
print("=" * 60)
print(f"Stratified reference (de-duplicated, Full): {strat_row['accuracy']*100:.2f}%")
if os.path.exists("loso_results.json"):
    with open("loso_results.json") as fh:
        base = json.load(fh)["summary"]["accuracy"]
    print(f"Baseline LOSO (Full, no mirror, with dupes): "
          f"{base['mean']*100:.2f}% +/- {base['std']*100:.2f}%")
print()
for name, a in results["arms"].items():
    print(f"  {name:<18} {a['mean_accuracy']*100:6.2f}% +/- {a['std_accuracy']*100:5.2f}%")

print("\nPer-fold accuracy by arm (%):")
signers_order = [r["held_out_signer"] for r in next(iter(results["arms"].values()))["folds"]]
print(f"{'Arm':<18}" + "".join(f"{s.replace('Signer_',''):>22}" for s in signers_order))
for name, a in results["arms"].items():
    print(f"{name:<18}" + "".join(f"{r['accuracy']*100:>21.2f}%" for r in a["folds"]))

In [ ]:
# ============================================================
# 9 -- Charts
# ============================================================
fig, ax = plt.subplots(figsize=(11, 5))
w = 0.8 / len(results["arms"])
xp = np.arange(len(signers_order))
for k, (name, a) in enumerate(results["arms"].items()):
    ax.bar(xp + (k - (len(results["arms"]) - 1) / 2) * w,
           [r["accuracy"] * 100 for r in a["folds"]], w, label=name)
ax.set_xticks(xp)
ax.set_xticklabels([s.replace("Signer_", "") for s in signers_order])
ax.set_ylabel("Accuracy (%)")
ax.set_title("LOSO Accuracy by Held-Out Signer and Feature Arm", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("phase1_loso_by_signer.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
names = list(results["arms"])
means = [results["arms"][n]["mean_accuracy"] * 100 for n in names]
stds = [results["arms"][n]["std_accuracy"] * 100 for n in names]
ax.bar(names, means, yerr=stds, capsize=5, color=ACCENT, edgecolor="white")
if os.path.exists("loso_results.json"):
    ax.axhline(base["mean"] * 100, color=RED, lw=1.5, linestyle="--",
               label=f"Phase 0 baseline {base['mean']*100:.2f}%")
    ax.legend()
ax.set_ylabel("Mean LOSO Accuracy (%)")
ax.set_title("Mean LOSO Accuracy by Arm (error bars = std across 4 signers)", fontweight="bold")
plt.tight_layout()
plt.savefig("phase1_arm_means.png", dpi=120)
plt.show()